In [1]:
from __future__ import annotations

import sys
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "common").exists():
        repository_root = str(candidate)
        if repository_root not in sys.path:
            sys.path.insert(0, repository_root)
        break

from notebooks.common.audit import add_audit_columns
from notebooks.common.metrics import (
    calculate_data_quality_score,
    create_pipeline_metric_df,
    print_summary,
)
from notebooks.common.validation import (
    apply_validation_rules,
    split_valid_and_quarantine,
)
from notebooks.config.northstar.paths import PATHS
from notebooks.config.northstar.schemas import ENROLLMENT_SCHEMA
from notebooks.config.northstar.validation_rules import (
    build_enrollment_validation_rules,
)

ModuleNotFoundError: No module named 'pyspark'

In [0]:
NOTEBOOK_VERSION = "1.1.0"
PIPELINE_NAME = "northstar_bronze_to_silver_enrollments"

BUSINESS_DATE = "2026-07-01"
BATCH_ID = "northstar-enrollments-20260701"
SOURCE_SYSTEM = "Enrollment Platform"
BRONZE_FILENAME = "enrollments_20260701.csv"

WRITE_MODE = "overwrite"

BRONZE_PATH = PATHS.bronze_file("enrollment/enrollments", BRONZE_FILENAME)
SILVER_PATH = PATHS.silver_path("enrollments")
QUARANTINE_PATH = PATHS.quarantine_path("enrollments")  # was pointing at gold — fixed
METRICS_PATH = PATHS.gold_path("data_quality_metrics")

print(BRONZE_PATH)
print(SILVER_PATH)

In [0]:
def read_bronze_enrollments(path: str) -> DataFrame:
    bronze_schema = StructType(
        [
            field
            for field in ENROLLMENT_SCHEMA.fields
            if field.name != "_corrupt_record"
        ]
        + [StructField("_corrupt_record", StringType(), nullable=True)]
    )

    return (
        spark.read.format("csv")
        .schema(bronze_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(path)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file_path"),
        )
    )


bronze_df = read_bronze_enrollments(BRONZE_PATH)

records_read = bronze_df.count()

print(f"Enrollment Bronze records read: {records_read:,}")

display(bronze_df.limit(10))

In [0]:
validated_df = apply_validation_rules(
    bronze_df,
    build_enrollment_validation_rules(as_of_date=BUSINESS_DATE),
)

valid_df, quarantine_df = split_valid_and_quarantine(validated_df)

valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {quarantine_count:,}")

display(quarantine_df.limit(10))

In [0]:
silver_df = add_audit_columns(
    valid_df,
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
)

quarantine_output_df = add_audit_columns(
    quarantine_df,
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
)

display(silver_df.limit(10))

In [0]:
(
    silver_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

silver_written_count = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
    .count()
)

print(f"Silver rows written: {silver_written_count:,}")

In [ ]:

(
    quarantine_output_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(QUARANTINE_PATH)
)

quarantine_written_count = (
    spark.read
    .format("delta")
    .load(QUARANTINE_PATH)
    .count()
)

print(f"Quarantine rows written: {quarantine_written_count:,}")

In [0]:
data_quality_score = calculate_data_quality_score(
    records_read,
    quarantine_count,
)

print_summary(
    records_read=records_read,
    records_written=silver_written_count,
    rejected_records=quarantine_written_count,
    data_quality_score=data_quality_score,
)